<a href="https://colab.research.google.com/github/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN/blob/main/04_train_pinn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===== CELL 1: KẾT NỐI DRIVE & KIỂM TRA GPU =====
from google.colab import drive  # Import thư viện drive của Colab để gắn kết nối tới Google Drive
import torch                    # Import thư viện PyTorch làm framework chính cho Deep Learning
import os                       # Import thư viện os để thực hiện các thao tác kiểm tra thư mục và đường dẫn

drive.mount('/content/drive')   # Thực hiện Mount (gắn) Google Drive của bạn vào môi trường ảo Colab

# Khai báo đường dẫn thư mục dự án gốc trên Google Drive
DRIVE_PROJECT = '/content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN'

# Khai báo đường dẫn chứa tập dữ liệu đã qua tiền xử lý (.npy và .pkl)
PROCESSED_DATA_DIR = os.path.join(DRIVE_PROJECT, 'dataset', 'processed')

# Khai báo thư mục lưu trữ trọng số mô hình tốt nhất (Checkpoints)
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT, 'checkpoints')

# Tự động tạo thư mục checkpoints trên Google Drive nếu thư mục này chưa tồn tại
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Kiểm tra xem Colab có GPU không: nếu có thì dùng 'cuda', nếu không thì dùng 'cpu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Thiết bị tính toán đang dùng:', device) # In ra màn hình thiết bị đang chạy (ví dụ: cuda:0)

Mounted at /content/drive
Thiết bị tính toán đang dùng: cpu


In [2]:
# ===== CELL 2: PYTORCH DATASET CHO PINN =====
from torch.utils.data import Dataset, DataLoader # Import hai lớp quản lý Dataset và nạp dữ liệu theo Batch
import numpy as np                               # Import NumPy để nạp các file lưu dưới dạng mảng .npy

class PMSMPINNDataset(Dataset):
    """
    Lớp Custom Dataset dùng để nạp Input (X), Target (Y) và Chuỗi thời gian (Time)
    phục vụ cho tính toán vi phân trong hàm Loss Vật Lý của PINN.
    """
    def __init__(self, data_dir, split='train'):
        # Nạp mảng Input đã chuẩn hóa: ud, uq, Id, Iq, nm
        self.X = np.load(os.path.join(data_dir, f'X_{split}.npy'))

        # Nạp mảng Target đã chuẩn hóa: Rs, Ld, Lq
        self.Y = np.load(os.path.join(data_dir, f'Y_{split}.npy'))

        # Nạp mảng chuỗi thời gian Time để hỗ trợ tính toán đạo hàm theo thời gian
        self.time = np.load(os.path.join(data_dir, f'Time_{split}.npy'))

        # Chuyển đổi dữ liệu từ dạng NumPy Array sang Tensor của PyTorch (kiểu float32)
        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.Y = torch.tensor(self.Y, dtype=torch.float32)
        self.time = torch.tensor(self.time, dtype=torch.float32)

    def __len__(self):
        # Trả về tổng số lượng mẫu dữ liệu có trong tập dữ liệu
        return len(self.X)

    def __getitem__(self, idx):
        # Trả về bộ 3 giá trị tương ứng với chỉ số (index) idx: Input, Target và Time
        return self.X[idx], self.Y[idx], self.time[idx]

# Nạp 3 tập dữ liệu đã phân chia từ thư mục processed trên Drive
train_dataset = PMSMPINNDataset(PROCESSED_DATA_DIR, split='train')       # Tập Train (Data1, Data2, Data3)
val_dataset = PMSMPINNDataset(PROCESSED_DATA_DIR, split='validation') # Tập Validation (Data4)
test_dataset = PMSMPINNDataset(PROCESSED_DATA_DIR, split='test')         # Tập Test (Data5)

# Khởi tạo DataLoader để chia dữ liệu thành từng Batch (kích thước 128 mẫu/batch)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)  # Trộn ngẫu nhiên tập Train khi nạp
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)     # Giữ nguyên thứ tự tập Validation
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)   # Giữ nguyên thứ tự tập Test

print(f'Đã nạp Dataset PINN thành công! Train: {len(train_dataset)} mẫu.')

Đã nạp Dataset PINN thành công! Train: 90000 mẫu.


In [8]:
# ===== CELL 3: KIẾN TRÚC MẠNG PINN & PHYSICS LOSS =====
import torch.nn as nn # Import module chứa các lớp mạng Neural của PyTorch
import joblib        # Import joblib để đọc file lưu bộ chuẩn hóa Scaler

# Nạp bộ chuẩn hóa Scaler đã lưu từ bước preprocessing để giải chuẩn hóa dữ liệu khi tính phương trình vật lý
X_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'X_scaler.pkl'))
Y_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'Y_scaler.pkl'))

# Chuyển các giá trị mean (trung bình) và scale (độ lệch chuẩn) của bộ Scaler thành PyTorch Tensor trên GPU
x_mean = torch.tensor(X_scaler.mean_, dtype=torch.float32).to(device)
x_scale = torch.tensor(X_scaler.scale_, dtype=torch.float32).to(device)
y_mean = torch.tensor(Y_scaler.mean_, dtype=torch.float32).to(device)
y_scale = torch.tensor(Y_scaler.scale_, dtype=torch.float32).to(device)

class PMSM_PINN(nn.Module):
    """
    Kiến trúc Mạng Feedforward Neural Network (FNN)
    Gồm 5 nút đầu vào [ud, uq, Id, Iq, nm] và 3 nút đầu ra dự đoán [Rs, Ld, Lq]
    """
    def __init__(self, input_dim=5, output_dim=3, hidden_dim=128):
        super(PMSM_PINN, self).__init__()
        # Định nghĩa chuỗi lớp liên kết đầy đủ (Fully Connected Layers)
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),   # Lớp đầu vào: 5 biến -> 128 nút ẩn
            nn.Tanh(),                          # Hàm kích hoạt Tanh tạo tính phi tuyến
            nn.Linear(hidden_dim, hidden_dim),  # Lớp ẩn thứ nhất: 128 -> 128 nút
            nn.Tanh(),                          # Hàm kích hoạt Tanh
            nn.Linear(hidden_dim, hidden_dim),  # Lớp ẩn thứ hai: 128 -> 128 nút
            nn.Tanh(),                          # Hàm kích hoạt Tanh
            nn.Linear(hidden_dim, output_dim)   # Lớp đầu ra: 128 -> 3 tham số [Rs, Ld, Lq]
        )

    def forward(self, x):
        # Hàm lan truyền tiến (Forward Pass) truyền dữ liệu qua mạng
        return self.network(x)

def compute_physics_loss(inputs_scaled, outputs_scaled, pole_pairs=4, psi_m=0.175, lambda_r=0.5):
    """
    Hàm tính toán Physics Loss tích hợp tư duy Tỷ số Ohm thuần trở (Rs = U_R / I)
    giúp ép Gradient của Rs hội tụ trực tiếp và loại bỏ nhiễu từ các độ tự cảm.
    """
    # 1. Unscale dữ liệu về dải thực tế
    inputs_real = inputs_scaled * x_scale + x_mean
    outputs_real = outputs_scaled * y_scale + y_mean

    ud = inputs_real[:, 0]
    uq = inputs_real[:, 1]
    Id = inputs_real[:, 2]
    Iq = inputs_real[:, 3]
    nm = inputs_real[:, 4]

    Rs_est = outputs_real[:, 0]
    Ld_est = outputs_real[:, 1]
    Lq_est = outputs_real[:, 2]

    # 2. Đổi tốc độ sang rad/s góc điện
    omega_m = nm * (2 * np.pi / 60.0)
    omega_e = pole_pairs * omega_m

    # 3. Physics Loss 1: Điện áp dư (Cũ)
    ud_calc = Rs_est * Id - omega_e * Lq_est * Iq
    uq_calc = Rs_est * Iq + omega_e * (Ld_est * Id + psi_m)

    loss_ud = torch.mean((ud - ud_calc) ** 2)
    loss_uq = torch.mean((uq - uq_calc) ** 2)
    loss_voltage = loss_ud + loss_uq

    # 4. Physics Loss 2: Tỷ số Ohm thuần trở Rs = (uq - BackEMF) / Iq (Đúng cơ sở bài báo)
    eps = 1e-4 # Tránh chia cho 0 khi dòng i_q rất nhỏ
    # Cô lập phần điện áp thuần trở trên trục q
    u_R_q_calc = uq - omega_e * (Ld_est * Id + psi_m)
    Rs_from_ohm = u_R_q_calc / (Iq + eps)

    # Thặng số trực tiếp cho Rs theo định luật Ohm
    loss_Rs_ohm = torch.mean((Rs_est - Rs_from_ohm) ** 2)

    # Tổng Physics Loss kết hợp
    return loss_voltage + lambda_r * loss_Rs_ohm

# Khởi tạo mô hình PINN và chuyển lên GPU
model_pinn = PMSM_PINN().to(device)
print(model_pinn) # In cấu trúc mạng Neural ra màn hình

PMSM_PINN(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Tanh()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): Tanh()
    (6): Linear(in_features=128, out_features=3, bias=True)
  )
)


In [9]:
# ===== CELL 4: HUẤN LUYỆN 2 GIAI ĐOẠN ĐÃ TỐI ƯU CHO RS =====
import torch.optim as optim

criterion_data = nn.MSELoss()
lambda_p = 1e-4
best_val_loss = float('inf')
checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_pinn_model.pth')

# ==========================================
# GIAI ĐOẠN 1: ADAM OPTIMIZER (100 Epochs)
# ==========================================
print("=== GIAI ĐOẠN 1: HUẤN LUYỆN BẰNG ADAM OPTIMIZER ===")
optimizer_adam = optim.Adam(model_pinn.parameters(), lr=0.001)
epochs_adam = 100

for epoch in range(1, epochs_adam + 1):
    model_pinn.train()
    train_data_loss, train_phys_loss, train_total_loss = 0.0, 0.0, 0.0

    for inputs, targets, _ in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer_adam.zero_grad()

        outputs = model_pinn(inputs)
        loss_d = criterion_data(outputs, targets)
        loss_p = compute_physics_loss(inputs, outputs)
        loss_total = loss_d + lambda_p * loss_p

        loss_total.backward()
        optimizer_adam.step()

        train_data_loss += loss_d.item() * inputs.size(0)
        train_phys_loss += loss_p.item() * inputs.size(0)
        train_total_loss += loss_total.item() * inputs.size(0)

    train_total_loss /= len(train_dataset)
    train_data_loss /= len(train_dataset)
    train_phys_loss /= len(train_dataset)

    # Validation check
    model_pinn.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets, _ in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model_pinn(inputs)
            loss_d = criterion_data(outputs, targets)
            val_loss += loss_d.item() * inputs.size(0)
    val_loss /= len(val_dataset)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch [{epoch:3d}/{epochs_adam}] | Data Loss: {train_data_loss:.6f} | Phys Loss: {train_phys_loss:.4f} | Val Loss: {val_loss:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_pinn.state_dict(),
            'val_loss': val_loss
        }, checkpoint_path)

# ==========================================
# GIAI ĐOẠN 2: TINH CHỈNH BẰNG L-BFGS OPTIMIZER (ĐÃ TỐI ƯU)
# ==========================================
print("\n=== GIAI ĐOẠN 2: TINH CHỈNH BẰNG L-BFGS OPTIMIZER ===")

# Nạp lại trọng số tốt nhất từ Giai đoạn 1
checkpoint = torch.load(checkpoint_path)
model_pinn.load_state_dict(checkpoint['model_state_dict'])

# Điều chỉnh lr = 0.01 nhỏ hơn để L-BFGS dò sâu vào đáy cực trị của Rs
optimizer_lbfgs = optim.LBFGS(
    model_pinn.parameters(),
    lr=0.01,
    max_iter=10,
    history_size=100,
    line_search_fn='strong_wolfe'
)

epochs_lbfgs = 20

for epoch in range(1, epochs_lbfgs + 1):
    model_pinn.train()

    def closure():
        optimizer_lbfgs.zero_grad()
        for inputs, targets, _ in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model_pinn(inputs)
            loss_d = criterion_data(outputs, targets)
            loss_p = compute_physics_loss(inputs, outputs)
            loss_total = loss_d + lambda_p * loss_p
            loss_total.backward()
            break
        return loss_total

    optimizer_lbfgs.step(closure)

    model_pinn.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets, _ in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model_pinn(inputs)
            loss_d = criterion_data(outputs, targets)
            val_loss += loss_d.item() * inputs.size(0)
    val_loss /= len(val_dataset)

    if epoch % 5 == 0 or epoch == 1:
        print(f"L-BFGS Epoch [{epoch:2d}/{epochs_lbfgs}] | Val Loss: {val_loss:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch + epochs_adam,
            'model_state_dict': model_pinn.state_dict(),
            'val_loss': val_loss
        }, checkpoint_path)

print(f"\n--> HOÀN THÀNH HUẤN LUYỆN 2 GIAI ĐOẠN! Weights tốt nhất lưu tại: {checkpoint_path}")

=== GIAI ĐOẠN 1: HUẤN LUYỆN BẰNG ADAM OPTIMIZER ===
Epoch [  1/100] | Data Loss: 0.300806 | Phys Loss: 270834.2318 | Val Loss: 0.364278
Epoch [ 10/100] | Data Loss: 0.247467 | Phys Loss: 270834.9624 | Val Loss: 0.314008
Epoch [ 20/100] | Data Loss: 0.208135 | Phys Loss: 270835.0442 | Val Loss: 0.289850
Epoch [ 30/100] | Data Loss: 0.193365 | Phys Loss: 270834.9969 | Val Loss: 0.297216
Epoch [ 40/100] | Data Loss: 0.179623 | Phys Loss: 270834.8836 | Val Loss: 0.324481
Epoch [ 50/100] | Data Loss: 0.168576 | Phys Loss: 270835.0098 | Val Loss: 0.290117
Epoch [ 60/100] | Data Loss: 0.153539 | Phys Loss: 270835.0556 | Val Loss: 0.372235
Epoch [ 70/100] | Data Loss: 0.145123 | Phys Loss: 270835.0263 | Val Loss: 0.295500
Epoch [ 80/100] | Data Loss: 0.127772 | Phys Loss: 270835.0340 | Val Loss: 0.297838
Epoch [ 90/100] | Data Loss: 0.129379 | Phys Loss: 270835.0178 | Val Loss: 0.296093
Epoch [100/100] | Data Loss: 0.116291 | Phys Loss: 270835.0195 | Val Loss: 0.263011

=== GIAI ĐOẠN 2: TINH C

In [10]:
# ===== CELL 5: ĐÁNH GIÁ PINN TRÊN TẬP TEST =====

# Nạp bản trọng số tốt nhất đã lưu trên Google Drive ra để đánh giá
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best_pinn_model.pth'))
model_pinn.load_state_dict(checkpoint['model_state_dict']) # Khôi phục trạng thái trọng số
model_pinn.eval() # Chuyển mô hình sang chế độ dự đoán/đánh giá

all_preds = []   # Mảng chứa toàn bộ kết quả dự đoán từ mạng PINN
all_targets = [] # Mảng chứa toàn bộ nhãn thực tế từ mô phỏng

with torch.no_grad(): # Tắt tính toán gradient khi kiểm tra tập Test
    for inputs, targets, _ in test_loader:
        inputs = inputs.to(device)                     # Đẩy dữ liệu lên GPU
        outputs = model_pinn(inputs)                  # Mạng PINN dự đoán
        all_preds.append(outputs.cpu().numpy())       # Lưu dự đoán về CPU dưới dạng NumPy
        all_targets.append(targets.numpy())           # Lưu nhãn thực tế dưới dạng NumPy

# Ghép các Batch nhỏ thành mảng 2D hoàn chỉnh
all_preds = np.vstack(all_preds)
all_targets = np.vstack(all_targets)

# Giải chuẩn hóa (Inverse Transform) kết quả dự đoán và nhãn thực tế về đơn vị thực (Ohm và Henry)
preds_real = Y_scaler.inverse_transform(all_preds)
targets_real = Y_scaler.inverse_transform(all_targets)

# Tính sai số phần trăm tuyệt đối trung bình (MAPE) cho từng tham số: Rs, Ld, Lq
mape = np.mean(np.abs((targets_real - preds_real) / targets_real), axis=0) * 100

print("=== KẾT QUẢ ĐÁNH GIÁ PINN TRÊN TẬP TEST ===")
print(f"Sai số Rs trung bình (PINN): {mape[0]:.2f}%") # In phần trăm sai số của Rs
print(f"Sai số Ld trung bình (PINN): {mape[1]:.2f}%") # In phần trăm sai số của Ld
print(f"Sai số Lq trung bình (PINN): {mape[2]:.2f}%") # In phần trăm sai số của Lq

=== KẾT QUẢ ĐÁNH GIÁ PINN TRÊN TẬP TEST ===
Sai số Rs trung bình (PINN): 4.72%
Sai số Ld trung bình (PINN): 0.11%
Sai số Lq trung bình (PINN): 0.03%
